In [1]:
"""
E-Commerce Data Analytics Pipeline - ETL & Data Modeling
Author: Nihat Rzaguluzada
Description: Extracts raw e-commerce sales data, cleans & transforms key features,
             performs feature engineering, and builds a Star Schema relational model
             (fact and dimension tables) ready for PostgreSQL & Power BI ingestion.
"""

import pandas as pd
import numpy as np

In [ ]:
# ==========================================
# 1. LOAD RAW DATA
# ==========================================

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\sales\ecommerce_sales_34500.csv")
df.head()

,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivery_time_days,region,returned,total_amount,shipping_cost,profit_margin,customer_age,customer_gender
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,2023-12-23,4,West,No,139.47,7.88,31.17,60,Female
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,2025-04-03,6,South,No,24.73,4.60,-2.62,37,Male
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,2024-10-08,4,North,No,166.80,6.58,13.44,34,Male
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,2024-09-14,6,South,No,63.67,5.50,2.14,21,Female
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,2024-12-21,6,East,No,13.88,2.74,1.15,39,Male


In [3]:
df.shape

(34500, 17)

In [4]:
df.isnull().sum()

order_id              0
customer_id           0
product_id            0
category              0
price                 0
discount              0
quantity              0
payment_method        0
order_date            0
delivery_time_days    0
region                0
returned              0
total_amount          0
shipping_cost         0
profit_margin         0
customer_age          0
customer_gender       0
dtype: int64

In [5]:
df.dtypes

order_id                  str
customer_id               str
product_id                str
category                  str
price                 float64
discount              float64
quantity                int64
payment_method            str
order_date                str
delivery_time_days      int64
region                    str
returned                  str
total_amount          float64
shipping_cost         float64
profit_margin         float64
customer_age            int64
customer_gender           str
dtype: object

In [2]:
# ==========================================
# 2. DATA CLEANING & TYPE CONVERSION
# ==========================================

In [7]:
# Convert order_date to datetime format
df['order_date'] = pd.to_datetime(df['order_date'])

In [8]:
df.dtypes

order_id                         str
customer_id                      str
product_id                       str
category                         str
price                        float64
discount                     float64
quantity                       int64
payment_method                   str
order_date            datetime64[us]
delivery_time_days             int64
region                           str
returned                         str
total_amount                 float64
shipping_cost                float64
profit_margin                float64
customer_age                   int64
customer_gender                  str
dtype: object

In [9]:
# Convert low-cardinality string columns to categorical dtypes for memory optimization
categorical_cols = ['category', 'payment_method', 'region', 'customer_gender']
for col in categorical_cols:
    df[col] = df[col].astype('category')

In [10]:
# Standardize boolean/string 'returned' column into a clean 1/0 integer flag
df['returned'] = df['returned'].astype(str).str.strip().str.lower().isin(['true', 'yes', '1']).astype(int)

In [11]:
df.dtypes

order_id                         str
customer_id                      str
product_id                       str
category                    category
price                        float64
discount                     float64
quantity                       int64
payment_method              category
order_date            datetime64[us]
delivery_time_days             int64
region                      category
returned                       int64
total_amount                 float64
shipping_cost                float64
profit_margin                float64
customer_age                   int64
customer_gender             category
dtype: object

In [12]:
df['returned'].unique()

array([0, 1])

In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================

In [13]:
# Binning customer age into strategic demographic groups
age_bins = [17, 25, 35, 50, 65, 100]
age_labels = ['18-25', '26-35', '36-50', '51-65', '65+']
df['age_group'] = pd.cut(df['customer_age'], bins=age_bins, labels=age_labels)

In [15]:
df['delivery_time_days'].unique()

array([ 4,  6,  5,  3,  7,  8,  9, 10, 13, 12])

In [16]:
# Categorizing delivery times into speed buckets
delivery_bins = [-1, 3, 7, 14]
delivery_labels = ['Fast (1-3 days)', 'Standard (4-7 days)', 'Delayed (8+ days)']
df['delivery_speed'] = pd.cut(df['delivery_time_days'], bins=delivery_bins, labels=delivery_labels)

In [17]:
# Financial Metrics Calculation
df['gross_amount'] = df['price'] * df['quantity']

In [18]:
df['discount_amount'] = df['gross_amount'] - df['total_amount']

In [19]:
# Account for returned products in net financial metrics
df['net_revenue'] = np.where(df['returned'] == 1, 0, df['total_amount'])
df['net_profit'] = np.where(df['returned'] == 1, 0, df['profit_margin'])

In [20]:
# Flag unprofitable orders
df['is_unprofitable'] = np.where(df['net_profit'] < 0, 1, 0)

In [21]:
df.head()

,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivery_time_days,...,profit_margin,customer_age,customer_gender,age_group,delivery_speed,gross_amount,discount_amount,net_revenue,net_profit,is_unprofitable
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,2023-12-23,4,...,31.17,60,Female,51-65,Standard (4-7 days),164.08,24.61,139.47,31.17,0
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,2025-04-03,6,...,-2.62,37,Male,36-50,Standard (4-7 days),24.73,0.00,24.73,-2.62,1
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,2024-10-08,4,...,13.44,34,Male,26-35,Standard (4-7 days),175.58,8.78,166.80,13.44,0
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,2024-09-14,6,...,2.14,21,Female,18-25,Standard (4-7 days),63.67,0.00,63.67,2.14,0
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,2024-12-21,6,...,1.15,39,Male,36-50,Standard (4-7 days),16.33,2.45,13.88,1.15,0


In [22]:
df.columns

Index(['order_id', 'customer_id', 'product_id', 'category', 'price',
       'discount', 'quantity', 'payment_method', 'order_date',
       'delivery_time_days', 'region', 'returned', 'total_amount',
       'shipping_cost', 'profit_margin', 'customer_age', 'customer_gender',
       'age_group', 'delivery_speed', 'gross_amount', 'discount_amount',
       'net_revenue', 'net_profit', 'is_unprofitable'],
      dtype='str')

In [23]:
df.to_csv("cleaned_ecommerce_sales.csv", index=False)

In [ ]:
# ==========================================
# 4. STAR SCHEMA DATA MODELING
# ==========================================

In [24]:
# --- Dimension Table 1: Customers ---
dim_customer = df[['customer_id', 'customer_age', 'customer_gender', 'age_group', 'region']].drop_duplicates(subset=['customer_id'])

In [25]:
dim_customer.head()

,customer_id,customer_age,customer_gender,age_group,region
0,C17270,60,Female,51-65,West
1,C17603,37,Male,36-50,South
2,C10860,34,Male,26-35,North
3,C15390,21,Female,18-25,South
4,C15226,39,Male,36-50,East


In [38]:
dim_customer.columns

Index(['customer_id', 'customer_age', 'customer_gender', 'age_group',
       'region'],
      dtype='str')

In [39]:
dim_customer.dtypes

customer_id             str
customer_age          int64
customer_gender    category
age_group          category
region             category
dtype: object

In [26]:
# --- Dimension Table 2: Products ---
dim_product = df[['product_id', 'category', 'price']].drop_duplicates(subset=['product_id'])

In [27]:
dim_product.head()

,product_id,category,price
0,P234890,Home,164.08
1,P228204,Grocery,24.73
2,P213892,Electronics,175.58
3,P208689,Electronics,63.67
4,P228063,Home,16.33


In [40]:
dim_product.columns

Index(['product_id', 'category', 'price'], dtype='str')

In [41]:
dim_product.dtypes

product_id         str
category      category
price          float64
dtype: object

In [35]:
# --- Dimension Table 3: Date Hierarchy ---
min_date = df['order_date'].min()
max_date = df['order_date'].max()
date_range = pd.date_range(start=min_date, end=max_date)

dim_date = pd.DataFrame({'date': date_range})
dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_of_week'] = dim_date['date'].dt.day_name()
dim_date['day_number_of_week'] = dim_date['date'].dt.dayofweek + 1
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek.isin([5, 6]).astype(int)

In [36]:
dim_date.head()

,date,year,quarter,month,month_name,day,day_of_week,day_number_of_week,is_weekend
0,2023-09-12,2023,Q3,9,September,12,Tuesday,2,0
1,2023-09-13,2023,Q3,9,September,13,Wednesday,3,0
2,2023-09-14,2023,Q3,9,September,14,Thursday,4,0
3,2023-09-15,2023,Q3,9,September,15,Friday,5,0
4,2023-09-16,2023,Q3,9,September,16,Saturday,6,1


In [42]:
dim_date.columns

Index(['date', 'year', 'quarter', 'month', 'month_name', 'day', 'day_of_week',
       'day_number_of_week', 'is_weekend'],
      dtype='str')

In [43]:
dim_date.dtypes

date                  datetime64[us]
year                           int32
quarter                          str
month                          int32
month_name                       str
day                            int32
day_of_week                      str
day_number_of_week             int32
is_weekend                     int64
dtype: object

In [31]:
# --- Fact Table: Sales Transactions ---
fact_sales = df[[
    'order_id', 'customer_id', 'product_id', 'order_date', 
    'payment_method', 'quantity', 'discount', 'gross_amount', 
    'discount_amount', 'total_amount', 'shipping_cost', 'profit_margin', 
    'net_revenue', 'net_profit', 'returned', 'is_unprofitable', 
    'delivery_time_days', 'delivery_speed'
]]

In [33]:
fact_sales.head()

,order_id,customer_id,product_id,order_date,payment_method,quantity,discount,gross_amount,discount_amount,total_amount,shipping_cost,profit_margin,net_revenue,net_profit,returned,is_unprofitable,delivery_time_days,delivery_speed
0,O100000,C17270,P234890,2023-12-23,Credit Card,1,0.15,164.08,24.61,139.47,7.88,31.17,139.47,31.17,0,0,4,Standard (4-7 days)
1,O100001,C17603,P228204,2025-04-03,Credit Card,1,0.00,24.73,0.00,24.73,4.60,-2.62,24.73,-2.62,0,1,6,Standard (4-7 days)
2,O100002,C10860,P213892,2024-10-08,Credit Card,1,0.05,175.58,8.78,166.80,6.58,13.44,166.80,13.44,0,0,4,Standard (4-7 days)
3,O100003,C15390,P208689,2024-09-14,UPI,1,0.00,63.67,0.00,63.67,5.50,2.14,63.67,2.14,0,0,6,Standard (4-7 days)
4,O100004,C15226,P228063,2024-12-21,COD,1,0.15,16.33,2.45,13.88,2.74,1.15,13.88,1.15,0,0,6,Standard (4-7 days)


In [44]:
fact_sales.columns

Index(['order_id', 'customer_id', 'product_id', 'order_date', 'payment_method',
       'quantity', 'discount', 'gross_amount', 'discount_amount',
       'total_amount', 'shipping_cost', 'profit_margin', 'net_revenue',
       'net_profit', 'returned', 'is_unprofitable', 'delivery_time_days',
       'delivery_speed'],
      dtype='str')

In [45]:
fact_sales.dtypes

order_id                         str
customer_id                      str
product_id                       str
order_date            datetime64[us]
payment_method              category
quantity                       int64
discount                     float64
gross_amount                 float64
discount_amount              float64
total_amount                 float64
shipping_cost                float64
profit_margin                float64
net_revenue                  float64
net_profit                   float64
returned                       int64
is_unprofitable                int64
delivery_time_days             int64
delivery_speed              category
dtype: object

In [37]:
# ==========================================
# 5. EXPORT PROCESSED DATASETS
# ==========================================
dim_customer.to_csv(r"C:\Users\ASUS\Desktop\sales\dim_customer.csv", index=False)
dim_product.to_csv(r"C:\Users\ASUS\Desktop\sales\dim_product.csv", index=False)
dim_date.to_csv(r"C:\Users\ASUS\Desktop\sales\dim_date.csv", index=False)
fact_sales.to_csv(r"C:\Users\ASUS\Desktop\sales\fact_sales.csv", index=False)